# GPT Model Architecture

This notebook implements the **full GPT decoder-only transformer** model:

- **TransformerBlock** — a single pre-norm decoder layer (GPT-2 style).
- **GPTModel** — the complete model: embeddings + stacked blocks + output head.

### Pre-Norm Transformer Architecture

Unlike the original Transformer ("post-norm"), GPT-2 applies LayerNorm **before**
each sub-layer rather than after. This is called **pre-norm** and improves training
stability, especially for smaller models.

Each TransformerBlock follows this pattern:

```
x  ──→  LN  ──→  Multi-Head Attention  ──→  + residual + dropout
   ──→  LN  ──→  Feed-Forward MLP      ──→  + residual + dropout
```

### GPT Model Assembly

The full model stacks N TransformerBlocks between an embedding layer and
a linear output head:

```
Token IDs  ──→  Token Embedding + Positional Embedding  ──→  Dropout
           ──→  TransformerBlock × N
           ──→  Final LayerNorm  ──→  Linear Head (→ vocab_size logits)
```

The positional embedding is a learned embedding (not sinusoidal), supporting
sequences up to `max_seq_len` tokens.

In [ ]:
import math
import sys

import torch
import torch.nn as nn
from torch import Tensor

## GPTConfig

All model hyperparameters flow through a single configuration dataclass.
The `validate()` method catches invalid values early — before any tensors
are allocated.

In [ ]:
from dataclasses import dataclass


@dataclass
class GPTConfig:
    """Configuration for the GPT model architecture."""

    vocab_size: int = 8000
    d_model: int = 256
    num_heads: int = 8
    num_layers: int = 6
    max_seq_len: int = 512
    dropout_rate: float = 0.1

    def validate(self) -> None:
        """Validate configuration values.

        Raises:
            ValueError: If d_model is not divisible by num_heads,
                        if any numeric value is non-positive,
                        or if dropout_rate is outside [0, 1].
        """
        if self.vocab_size <= 0:
            raise ValueError(f"vocab_size must be positive, got {self.vocab_size}")
        if self.d_model <= 0:
            raise ValueError(f"d_model must be positive, got {self.d_model}")
        if self.num_heads <= 0:
            raise ValueError(f"num_heads must be positive, got {self.num_heads}")
        if self.num_layers <= 0:
            raise ValueError(f"num_layers must be positive, got {self.num_layers}")
        if self.max_seq_len <= 0:
            raise ValueError(f"max_seq_len must be positive, got {self.max_seq_len}")
        if self.dropout_rate < 0 or self.dropout_rate > 1:
            raise ValueError(
                f"dropout_rate must be between 0 and 1, got {self.dropout_rate}"
            )
        if self.d_model % self.num_heads != 0:
            raise ValueError(
                f"d_model ({self.d_model}) must be divisible by num_heads ({self.num_heads})"
            )

## Attention Modules (from previous notebook)

We need `ScaledDotProductAttention` and `MultiHeadAttention` as building
blocks. These are defined in `04_attention.ipynb` and `src/model/attention.py`.
We reproduce them here so this notebook is self-contained.

In [ ]:
class ScaledDotProductAttention(nn.Module):
    """Scaled dot-product attention with causal masking."""

    def forward(self, Q: Tensor, K: Tensor, V: Tensor, mask: Tensor | None = None) -> Tensor:
        d_k = Q.size(-1)
        seq_len = Q.size(-2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is None:
            mask = torch.triu(
                torch.ones(seq_len, seq_len, device=Q.device, dtype=torch.bool),
                diagonal=1,
            )
        scores = scores.masked_fill(mask, float("-inf"))
        attn_weights = torch.softmax(scores, dim=-1)
        return torch.matmul(attn_weights, V)


class MultiHeadAttention(nn.Module):
    """Multi-head attention module."""

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(
                f"d_model ({d_model}) must be evenly divisible by num_heads ({num_heads})"
            )
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.attention = ScaledDotProductAttention()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: Tensor) -> Tensor:
        batch_size, seq_len, _ = x.size()
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        attn_output = self.attention(Q, K, V)
        attn_output = self.dropout(attn_output)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_o(attn_output)

## TransformerBlock

A single pre-norm decoder block. The architecture is:

1. **LayerNorm → Multi-Head Attention → Residual + Dropout**
2. **LayerNorm → Feed-Forward MLP → Residual + Dropout**

### Pre-Norm vs Post-Norm

In **post-norm** (original Transformer), LayerNorm is applied *after* the
residual addition: `x = LN(x + sublayer(x))`. In **pre-norm** (GPT-2),
LayerNorm is applied *before* the sub-layer: `x = x + sublayer(LN(x))`.

Pre-norm keeps the residual stream "clean" — gradients flow directly through
the residual connections without passing through normalization layers, which
stabilizes training.

### Feed-Forward MLP

The MLP expands the dimension by 4× then projects back down:

```
d_model  ──→  Linear(d_model, 4*d_model)  ──→  GELU  ──→  Linear(4*d_model, d_model)
```

GELU (Gaussian Error Linear Unit) is the activation used in GPT-2, providing
a smooth approximation to ReLU.

### Residual Connections

Each sub-layer output is added back to its input (`x = x + sublayer(...)`).
This allows gradients to flow directly through the network and enables
training of deep models. Dropout is applied to the sub-layer output before
the residual addition.

| Component | Input Shape | Output Shape |
|---|---|---|
| LayerNorm 1 | `(B, S, d_model)` | `(B, S, d_model)` |
| Multi-Head Attention | `(B, S, d_model)` | `(B, S, d_model)` |
| LayerNorm 2 | `(B, S, d_model)` | `(B, S, d_model)` |
| MLP Linear 1 | `(B, S, d_model)` | `(B, S, 4*d_model)` |
| GELU | `(B, S, 4*d_model)` | `(B, S, 4*d_model)` |
| MLP Linear 2 | `(B, S, 4*d_model)` | `(B, S, d_model)` |

In [ ]:
class TransformerBlock(nn.Module):
    """Pre-norm transformer decoder block.

    Architecture: LN → MHA → residual + dropout → LN → MLP → residual + dropout.
    The MLP uses GELU activation with inner dimension = 4 * d_model.
    """

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        """Initialize TransformerBlock.

        Args:
            d_model: Model dimension.
            num_heads: Number of attention heads.
            dropout: Dropout rate applied after attention and MLP outputs.
        """
        super().__init__()

        # Pre-norm layers
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        # Multi-head attention
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)

        # Feed-forward MLP: linear → GELU → linear
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )

        # Dropout for residual connections
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: Tensor) -> Tensor:
        """Forward pass through the transformer block.

        Args:
            x: Input tensor of shape (batch, seq_len, d_model).

        Returns:
            Output tensor of shape (batch, seq_len, d_model).
        """
        # LN → MHA → residual + dropout
        x = x + self.dropout1(self.attn(self.ln1(x)))

        # LN → MLP → residual + dropout
        x = x + self.dropout2(self.mlp(self.ln2(x)))

        return x

## GPTModel

The full decoder-only transformer model. It assembles all components:

1. **Token Embedding** — maps each token ID to a `d_model`-dimensional vector.
2. **Positional Embedding** — learned embeddings for each position `0..max_seq_len-1`.
   Token and positional embeddings are summed, then dropout is applied.
3. **N Stacked TransformerBlocks** — the core of the model. Each block refines
   the representations through attention and feed-forward processing.
4. **Final LayerNorm** — applied after the last block (pre-norm convention).
5. **Linear Head** — projects from `d_model` to `vocab_size` to produce logits
   for next-token prediction.

### Sequence Length Validation

The positional embedding table has exactly `max_seq_len` entries. If the input
sequence is longer, the model raises a `ValueError` rather than silently
producing garbage outputs.

### Parameter Count

The `count_parameters()` method returns the total number of trainable parameters.
For the default config (d_model=256, 6 layers, 8 heads, vocab=8000):

- Token embedding: `vocab_size × d_model`
- Positional embedding: `max_seq_len × d_model`
- Per block: attention projections + MLP + LayerNorms
- Output head: `d_model × vocab_size`

In [ ]:
class GPTModel(nn.Module):
    """Full GPT decoder-only transformer model.

    Assembles: token embedding + positional embedding + N TransformerBlocks
    + final LayerNorm + linear head (→ vocab_size).
    """

    def __init__(self, config: GPTConfig):
        """Initialize GPTModel from a GPTConfig.

        Args:
            config: GPTConfig with vocab_size, d_model, num_heads,
                    num_layers, max_seq_len, dropout_rate.
        """
        super().__init__()
        config.validate()
        self.config = config

        # Token and positional embeddings
        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.max_seq_len, config.d_model)

        # Dropout after embeddings
        self.drop = nn.Dropout(config.dropout_rate)

        # N stacked transformer blocks
        self.blocks = nn.ModuleList(
            [
                TransformerBlock(config.d_model, config.num_heads, config.dropout_rate)
                for _ in range(config.num_layers)
            ]
        )

        # Final layer norm and linear head
        self.ln_f = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size)

    def forward(self, token_ids: Tensor) -> Tensor:
        """Forward pass: token IDs → logits.

        Args:
            token_ids: Input tensor of shape (batch, seq_len) with integer token IDs.

        Returns:
            Logits tensor of shape (batch, seq_len, vocab_size).

        Raises:
            ValueError: If seq_len exceeds config.max_seq_len.
        """
        _, seq_len = token_ids.size()

        if seq_len > self.config.max_seq_len:
            raise ValueError(
                f"Input sequence length {seq_len} exceeds maximum supported "
                f"length {self.config.max_seq_len}"
            )

        # Create position indices: (seq_len,)
        positions = token_ids.new_tensor(range(seq_len), dtype=token_ids.dtype)

        # Embeddings: token + positional
        x = self.token_emb(token_ids) + self.pos_emb(positions)
        x = self.drop(x)

        # Pass through transformer blocks
        for block in self.blocks:
            x = block(x)

        # Final layer norm + linear head
        x = self.ln_f(x)
        logits = self.lm_head(x)

        return logits

    def count_parameters(self) -> int:
        """Return total number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

---

## Demos

The cells below demonstrate model instantiation, parameter counting,
a forward pass, and sequence length validation.

### Demo: Model Instantiation and Parameter Count

Create a GPTModel with the default config and inspect its structure
and total trainable parameter count.

In [ ]:
config = GPTConfig()
model = GPTModel(config)

print(f"GPTConfig:")
print(f"  vocab_size   = {config.vocab_size}")
print(f"  d_model      = {config.d_model}")
print(f"  num_heads    = {config.num_heads}")
print(f"  num_layers   = {config.num_layers}")
print(f"  max_seq_len  = {config.max_seq_len}")
print(f"  dropout_rate = {config.dropout_rate}")
print()
print(f"Total trainable parameters: {model.count_parameters():,}")
print()
print("Model structure:")
print(model)

### Demo: Forward Pass with Random Token IDs

Feed a batch of random token IDs through the model and verify the output
shape is `(batch, seq_len, vocab_size)`.

In [ ]:
torch.manual_seed(42)

batch_size = 2
seq_len = 16

# Random token IDs in [0, vocab_size)
token_ids = torch.randint(0, config.vocab_size, (batch_size, seq_len))

model.eval()
with torch.no_grad():
    logits = model(token_ids)

print(f"Input shape:  {token_ids.shape}  (batch={batch_size}, seq_len={seq_len})")
print(f"Output shape: {logits.shape}  (batch={batch_size}, seq_len={seq_len}, vocab_size={config.vocab_size})")
print(f"Shape correct: {logits.shape == (batch_size, seq_len, config.vocab_size)}")
print()
print(f"Sample logits for first token in first sequence:")
print(f"  min={logits[0, 0].min().item():.4f}, max={logits[0, 0].max().item():.4f}, mean={logits[0, 0].mean().item():.4f}")

### Demo: Sequence Length Exceeds max_seq_len

If the input sequence is longer than `max_seq_len`, the model raises a
`ValueError` immediately rather than producing incorrect outputs.

In [ ]:
# Create a sequence longer than max_seq_len (512)
too_long = torch.randint(0, config.vocab_size, (1, config.max_seq_len + 1))

try:
    model(too_long)
    print("ERROR: Should have raised ValueError!")
except ValueError as e:
    print(f"Caught expected error: {e}")
    print(f"\nInput seq_len={too_long.shape[1]}, max_seq_len={config.max_seq_len}")